# 03 — Query Iceberg Tables

**Pre-requisites:**
- Run `01_csv_to_iceberg.ipynb` to populate `iceberg_catalog.my_database.orders`
- Run `02_streaming_to_iceberg.ipynb` (with producer) to populate `iceberg_catalog.my_database.user_events`

**What this notebook covers:**
- Spark SQL queries on both tables
- DataFrame API with aggregations and joins
- Iceberg time-travel and snapshot inspection

In [ ]:
from app.utils.spark_session import spark, display_df
from pyspark.sql import functions as F

spark

## Discover tables

In [2]:
spark.sql("SHOW TABLES IN iceberg_catalog.my_database").show(truncate=False)

+-----------+--------------+-----------+
|namespace  |tableName     |isTemporary|
+-----------+--------------+-----------+
|my_database|user_events   |false      |
|my_database|orders_iceberg|false      |
+-----------+--------------+-----------+



---
## Table 1 — `orders` (batch / CSV source)

In [4]:
# Full table scan — SQL
spark.sql("SELECT * FROM iceberg_catalog.my_database.orders_iceberg ORDER BY order_id").show(truncate=False)

+--------+-------------+------------+--------+----------+----------+---------+
|order_id|customer_name|product     |quantity|unit_price|order_date|status   |
+--------+-------------+------------+--------+----------+----------+---------+
|1       |Alice Johnson|Laptop      |1       |1299.99   |2024-01-15|completed|
|2       |Bob Smith    |Mouse       |2       |29.99     |2024-01-16|completed|
|3       |Carol White  |Keyboard    |1       |79.99     |2024-01-17|pending  |
|4       |David Lee    |Monitor     |1       |399.99    |2024-01-18|completed|
|5       |Eve Martinez |Headphones  |3       |149.99    |2024-01-19|shipped  |
|6       |Frank Brown  |Webcam      |2       |89.99     |2024-01-20|pending  |
|7       |Grace Kim    |USB Hub     |1       |49.99     |2024-01-21|completed|
|8       |Henry Wilson |Laptop Stand|2       |59.99     |2024-01-22|shipped  |
|9       |Iris Chen    |SSD Drive   |1       |119.99    |2024-01-23|completed|
|10      |James Davis  |Microphone  |1       |199.99

In [6]:
# Revenue by status — SQL
spark.sql("""
    SELECT
        status,
        COUNT(*)                             AS orders,
        SUM(quantity)                        AS units_sold,
        ROUND(SUM(quantity * unit_price), 2) AS revenue
    FROM iceberg_catalog.my_database.orders_iceberg
    GROUP BY status
    ORDER BY revenue DESC
""").show()

+---------+------+----------+-------+
|   status|orders|units_sold|revenue|
+---------+------+----------+-------+
|completed|     5|         6|1929.94|
|  shipped|     2|         5| 569.95|
|  pending|     3|         4| 459.96|
+---------+------+----------+-------+



In [7]:
# Top products by revenue — DataFrame API
orders_df = spark.table("iceberg_catalog.my_database.orders_iceberg")

(
    orders_df
    .withColumn("revenue", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .groupBy("product")
    .agg(
        F.sum("revenue").alias("total_revenue"),
        F.sum("quantity").alias("total_units"),
        F.countDistinct("customer_name").alias("unique_customers"),
    )
    .orderBy(F.desc("total_revenue"))
    .show(truncate=False)
)

+------------+-------------+-----------+----------------+
|product     |total_revenue|total_units|unique_customers|
+------------+-------------+-----------+----------------+
|Laptop      |1299.99      |1          |1               |
|Headphones  |449.97       |3          |1               |
|Monitor     |399.99       |1          |1               |
|Microphone  |199.99       |1          |1               |
|Webcam      |179.98       |2          |1               |
|SSD Drive   |119.99       |1          |1               |
|Laptop Stand|119.98       |2          |1               |
|Keyboard    |79.99        |1          |1               |
|Mouse       |59.98        |2          |1               |
|USB Hub     |49.99        |1          |1               |
+------------+-------------+-----------+----------------+



---
## Table 2 — `user_events` (streaming / JSON source)

In [8]:
# Full table scan — SQL
spark.sql("""
    SELECT * FROM iceberg_catalog.my_database.user_events
    ORDER BY timestamp DESC
    LIMIT 20
""").show(truncate=False)

+------------------------------------+--------+-----------+---------+--------------------------------+----------+
|event_id                            |user_id |event_type |page     |timestamp                       |session_id|
+------------------------------------+--------+-----------+---------+--------------------------------+----------+
|a3245283-32ef-4cfd-9f35-4584be0ba542|user_019|add_to_cart|/products|2026-04-28T19:46:24.935037+00:00|7977797b  |
|f560e4ae-ec3d-49b4-9432-cc88fc465e02|user_003|click      |/products|2026-04-28T19:46:24.934979+00:00|c27e6282  |
|d19ef6d1-c0ba-4f49-89d6-03d837f31745|user_016|add_to_cart|/orders  |2026-04-28T19:46:14.930903+00:00|4401ba11  |
|ff2e2857-1f68-42d5-9526-32011cbacc1c|user_007|purchase   |/home    |2026-04-28T19:46:14.930861+00:00|4e22c43c  |
|b6c5a50f-8beb-4e8e-9af1-59618b2864c7|user_019|logout     |/checkout|2026-04-28T19:46:04.926465+00:00|39618007  |
|c51fa56e-fc16-4698-9cd1-75c3d1b56d3e|user_014|search     |/home    |2026-04-28T19:46:04

In [9]:
# Event breakdown — SQL
spark.sql("""
    SELECT
        event_type,
        COUNT(*)                    AS event_count,
        COUNT(DISTINCT user_id)     AS unique_users,
        COUNT(DISTINCT session_id)  AS unique_sessions
    FROM iceberg_catalog.my_database.user_events
    GROUP BY event_type
    ORDER BY event_count DESC
""").show()

+-----------+-----------+------------+---------------+
| event_type|event_count|unique_users|unique_sessions|
+-----------+-----------+------------+---------------+
|     search|         26|          13|             26|
|      click|         26|          13|             26|
|add_to_cart|         25|          13|             25|
|   purchase|         25|          13|             25|
|  page_view|         25|          11|             25|
|     logout|         13|          10|             13|
+-----------+-----------+------------+---------------+



In [10]:
# Top active users — DataFrame API
events_df = spark.table("iceberg_catalog.my_database.user_events")

(
    events_df
    .groupBy("user_id")
    .agg(
        F.count("*").alias("total_events"),
        F.countDistinct("event_type").alias("distinct_event_types"),
        F.countDistinct("page").alias("pages_visited"),
    )
    .orderBy(F.desc("total_events"))
    .show(10)
)

+--------+------------+--------------------+-------------+
| user_id|total_events|distinct_event_types|pages_visited|
+--------+------------+--------------------+-------------+
|user_007|          14|                   6|            6|
|user_019|          13|                   5|            7|
|user_012|          12|                   5|            7|
|user_003|          11|                   5|            6|
|user_006|          10|                   5|            5|
|user_020|           9|                   4|            5|
|user_009|           9|                   4|            3|
|user_002|           9|                   5|            5|
|user_014|           7|                   3|            6|
|user_013|           6|                   4|            2|
+--------+------------+--------------------+-------------+
only showing top 10 rows


In [11]:
# Most visited pages — DataFrame API
(
    events_df
    .groupBy("page")
    .agg(F.count("*").alias("visits"))
    .orderBy(F.desc("visits"))
    .show()
)

+---------+------+
|     page|visits|
+---------+------+
|/checkout|    30|
|/products|    26|
|  /orders|    21|
| /profile|    18|
|    /home|    16|
|  /search|    15|
|    /cart|    14|
+---------+------+



---
## Cross-table analysis

Find users who placed orders **and** had streaming events  
(works once `customer_name` and `user_id` overlap, e.g. `user_001` etc.)

In [15]:
# Purchase events alongside order revenue per customer — SQL
spark.sql("""
    SELECT
        e.user_id,
        COUNT(e.event_id)                        AS total_events,
        SUM(CASE WHEN e.event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_events,
        COUNT(DISTINCT e.page)                   AS pages_visited
    FROM iceberg_catalog.my_database.user_events e
    GROUP BY e.user_id
    ORDER BY purchase_events DESC, total_events DESC
""").show(10)

+--------+------------+---------------+-------------+
| user_id|total_events|purchase_events|pages_visited|
+--------+------------+---------------+-------------+
|user_006|          10|              4|            5|
|user_020|           9|              4|            5|
|user_012|          12|              3|            7|
|user_007|          14|              2|            6|
|user_003|          11|              2|            6|
|user_015|           6|              2|            4|
|user_008|           6|              2|            4|
|user_002|           9|              1|            5|
|user_014|           7|              1|            6|
|user_016|           6|              1|            5|
+--------+------------+---------------+-------------+
only showing top 10 rows


---
## Iceberg metadata & time travel

In [20]:
# Snapshots for orders
print("=== orders snapshots ===")
spark.sql("""
    SELECT snapshot_id, committed_at, operation, summary
    FROM iceberg_catalog.my_database.orders_iceberg.snapshots
""").show(truncate=False)

# Snapshots for user_events
print("=== user_events snapshots ===")
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM iceberg_catalog.my_database.user_events.snapshots
    ORDER BY committed_at
""").show(20, truncate=False)

=== orders snapshots ===
+-------------------+-----------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |committed_at           |operation|summary                                                                                                                                                                                                                                                                                                                                                                                                            

In [21]:
# Time travel — read orders at a specific snapshot
snapshots = spark.sql("""
    SELECT snapshot_id FROM iceberg_catalog.my_database.orders_iceberg.snapshots ORDER BY committed_at
""").collect()

if snapshots:
    first_snapshot = snapshots[0]["snapshot_id"]
    print(f"Reading orders at snapshot: {first_snapshot}")
    spark.read \
         .option("snapshot-id", first_snapshot) \
         .table("iceberg_catalog.my_database.orders_iceberg") \
         .show(truncate=False)
else:
    print("No snapshots found — run 01_csv_to_iceberg.ipynb first.")

Reading orders at snapshot: 1547481985620889591
+--------+-------------+------------+--------+----------+----------+---------+
|order_id|customer_name|product     |quantity|unit_price|order_date|status   |
+--------+-------------+------------+--------+----------+----------+---------+
|1       |Alice Johnson|Laptop      |1       |1299.99   |2024-01-15|completed|
|2       |Bob Smith    |Mouse       |2       |29.99     |2024-01-16|completed|
|3       |Carol White  |Keyboard    |1       |79.99     |2024-01-17|pending  |
|4       |David Lee    |Monitor     |1       |399.99    |2024-01-18|completed|
|5       |Eve Martinez |Headphones  |3       |149.99    |2024-01-19|shipped  |
|6       |Frank Brown  |Webcam      |2       |89.99     |2024-01-20|pending  |
|7       |Grace Kim    |USB Hub     |1       |49.99     |2024-01-21|completed|
|8       |Henry Wilson |Laptop Stand|2       |59.99     |2024-01-22|shipped  |
|9       |Iris Chen    |SSD Drive   |1       |119.99    |2024-01-23|completed|
|10 

In [22]:
# Data files layout — Iceberg metadata tables
print("=== orders data files ===")
spark.sql("""
    SELECT file_path, file_format, record_count, file_size_in_bytes
    FROM iceberg_catalog.my_database.orders_iceberg.files
""").show(truncate=False)

print("=== user_events data files ===")
spark.sql("""
    SELECT file_path, file_format, record_count, file_size_in_bytes
    FROM iceberg_catalog.my_database.user_events.files
""").show(truncate=False)

=== orders data files ===
+----------------------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+
|file_path                                                                                                                   |file_format|record_count|file_size_in_bytes|
+----------------------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+
|.tmp/local_iceberg_warehouse/my_database/orders_iceberg/data/00000-1301-7e038501-a746-4f44-876b-0982794eba95-0-00001.parquet|PARQUET    |10          |2520              |
+----------------------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+

=== user_events data files ===
+--------------------------------------------------------------------------------------